In [2]:
import polars as pl

# 1. Load Data (Using ../ to go up from the notebooks folder)
df_tx = pl.scan_csv('../Data/raw/transactions.csv')
df_cust = pl.scan_csv('../Data/raw/customers.csv')
df_prod = pl.scan_csv('../Data/raw/products.csv')
df_inv = pl.scan_csv('../Data/raw/inventory.csv')

# 2. Standardize Dates for Joining
df_tx = df_tx.with_columns(
    pl.col("timestamp").str.to_datetime().dt.date().alias("join_date")
)
df_inv = df_inv.with_columns(
    pl.col("date").str.to_datetime().dt.date().alias("join_date")
)

# 3. Execute the Multithreaded Master Join
df_master = (
    df_tx
    .join(df_cust, on="customer_id", how="left")
    .join(df_prod, on="product_id", how="left")
    .join(df_inv.drop("date"), on=["product_id", "join_date"], how="left")
).collect() 

print(f"Master Dataset Shape: {df_master.shape}")

Master Dataset Shape: (18027, 24)


In [3]:
# 1. Drop 'Pending' rows (unknown final outcome)
df_model = df_master.filter(pl.col("status") != "Pending")

# 2. Create Binary Target (1 = Leak/Cancelled/Returned, 0 = Completed)
df_model = df_model.with_columns(
    pl.when(pl.col("status").is_in(["Cancelled", "Returned"]))
    .then(1)
    .otherwise(0)
    .alias("target_leak")
)

# 3. Rule 07: Time-Based Split (Split at Nov 1, 2025)
train_date_cutoff = pl.date(2025, 11, 1)

train_data = df_model.filter(pl.col("join_date") < train_date_cutoff)
test_data = df_model.filter(pl.col("join_date") >= train_date_cutoff)

# 4. Drop non-predictive IDs/Dates and convert to Pandas for XGBoost
features_to_drop = ["transaction_id", "customer_id", "product_id", "inventory_record_id", "timestamp", "join_date", "status", "date"]

# Use list comprehension to only drop columns that actually exist in the dataframe
cols_to_drop = [col for col in features_to_drop if col in df_model.columns]

X_train = train_data.drop(cols_to_drop + ["target_leak"]).to_pandas()
y_train = train_data["target_leak"].to_pandas()

X_test = test_data.drop(cols_to_drop + ["target_leak"]).to_pandas()
y_test = test_data["target_leak"].to_pandas()

print(f"Training Data Shape: {X_train.shape}")
print(f"Testing Data Shape: {X_test.shape}")
print("\nTraining Target Distribution:")
print(y_train.value_counts())

Training Data Shape: (14308, 17)
Testing Data Shape: (2800, 17)

Training Target Distribution:
target_leak
0    11698
1     2610
Name: count, dtype: int64


In [4]:
import xgboost as xgb
import joblib
import os
from sklearn.metrics import classification_report

# 1. Convert string columns to categorical for XGBoost
cat_cols = X_train.select_dtypes(include=['object']).columns
for col in cat_cols:
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')

# 2. Calculate class imbalance weight (Negative class / Positive class)
imbalance_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])

# 3. Initialize and Train XGBoost Classifier
model = xgb.XGBClassifier(
    enable_categorical=True,
    tree_method="hist",
    scale_pos_weight=imbalance_weight,
    max_depth=5,
    learning_rate=0.1,
    n_estimators=100,
    random_state=42
)

print("Training XGBoost model from scratch...")
model.fit(X_train, y_train)

# 4. Evaluate on the strict Time-Based Test Set
y_pred = model.predict(X_test)
print("\n--- Test Set Performance (Rule 06 & 09) ---")
print(classification_report(y_test, y_pred))

# 5. Export binary snapshot for FastAPI backend
os.makedirs('../models', exist_ok=True)
joblib.dump(model, '../models/xgboost_v1.joblib')
print("\nModel successfully saved to ../models/xgboost_v1.joblib")

Training XGBoost model from scratch...

--- Test Set Performance (Rule 06 & 09) ---
              precision    recall  f1-score   support

           0       0.81      0.69      0.74      2285
           1       0.18      0.30      0.22       515

    accuracy                           0.62      2800
   macro avg       0.49      0.49      0.48      2800
weighted avg       0.70      0.62      0.65      2800


Model successfully saved to ../models/xgboost_v1.joblib


In [5]:
# Save column metadata and default values for the FastAPI inference engine
cat_cols = X_train.select_dtypes(include=['category']).columns.tolist()
num_cols = X_train.select_dtypes(exclude=['category']).columns.tolist()

meta = {
    "features": X_train.columns.tolist(),
    "cat_cols": cat_cols,
    "categories": {col: X_train[col].cat.categories.tolist() for col in cat_cols},
    "num_medians": {col: float(X_train[col].median()) for col in num_cols}
}

joblib.dump(meta, '../models/model_meta.joblib')
print("Metadata saved! Features:", meta["features"])

Metadata saved! Features: ['quantity', 'unit_price', 'discount_pct', 'channel', 'payment_method', 'region', 'customer_segment', 'signup_date', 'preferred_channel', 'category', 'unit_cost', 'list_price', 'supplier_lead_days', 'stock_on_hand', 'reorder_threshold', 'stockout_flag', 'supplier_delay_days']


In [6]:
import sqlite3
import pandas as pd
import os

os.makedirs('../data', exist_ok=True)
conn = sqlite3.connect('../data/money_leak.db')

# Load the 5 CSVs into local SQL tables
pd.read_csv('../Data/raw/transactions.csv').to_sql('transactions', conn, if_exists='replace', index=False)
pd.read_csv('../Data/raw/customers.csv').to_sql('customers', conn, if_exists='replace', index=False)
pd.read_csv('../Data/raw/products.csv').to_sql('products', conn, if_exists='replace', index=False)
pd.read_csv('../Data/raw/inventory.csv').to_sql('inventory', conn, if_exists='replace', index=False)
pd.read_csv('../Data/raw/campaigns.csv').to_sql('campaigns', conn, if_exists='replace', index=False)

# Test a relational join query
query_test = pd.read_sql("""
    SELECT t.transaction_id, t.status, c.region, p.category, t.unit_price
    FROM transactions t
    LEFT JOIN customers c ON t.customer_id = c.customer_id
    LEFT JOIN products p ON t.product_id = p.product_id
    LIMIT 5
""", conn)

print("✅ SQLite Database created successfully at ../data/money_leak.db")
print(query_test)
conn.close()

✅ SQLite Database created successfully at ../data/money_leak.db
  transaction_id     status   region     category  unit_price
0       TX000001  Completed     West  Electronics      307.70
1       TX000002  Completed    North         Home      346.98
2       TX000003   Returned     East      Grocery       33.87
3       TX000004    Pending  Central      Grocery      166.19
4       TX000005  Completed     East         Home      361.56
